# Marketplace Data Cleaning

This notebook prepares the marketplace dataset for downstream fragrance analysis.

The cleaning process focuses on identifying and excluding marketplace listings that do not represent a standard individual fragrance product.

## Objectives

- Identify non-standard marketplace listings
- Exclude wholesale and bulk listings
- Exclude bundles, sets, and multi-product collections
- Exclude empty and display bottles
- Exclude artwork and other non-fragrance merchandise
- Preserve an audit trail of excluded listings
- Create a clean analytical dataset for downstream analysis

The original marketplace data remains unchanged in `marketplace_base`.

The cleaned dataset will be used as the input for subsequent brand, price, gender, and other analyses.

In [0]:
# Marketplace Data Cleaning — load base dataset

from pyspark.sql import functions as F

# Load the immutable raw marketplace dataset
df = spark.table("main.fragrance_analytics.marketplace_base")

# Display the schema and a sample of the raw data
df.printSchema()
display(df.limit(20))

In [0]:
# Define and validate exclusion rules

# Generic brand values that do not identify an actual fragrance brand.
# These are only used for the explicitly reviewed listings, not as a blanket exclusion rule for the entire dataset.
generic_brand_values = [
    "assorted",
    "eau de parfum"
]

# Listings of reviewed and confirmed to be genuinely unidentifiable because they do not identify a specific fragrance brand.
previously_reviewed_unidentifiable_ids = [
    1001,
    1026,
    1027,
    1071,
    1075,
    1111,
    1143,
    1225,
    1397,
    1442,
    1588,
    1619,
    1667,
    2012,
    2017,
    2591,
    2869
]

# Wholesale listings represent commercial bulk sales rather than individual marketplace fragrance products.
wholesale_pattern = r"(?i)\bwholesale\b"

# Identify listings that should be excluded and assign an exclusion reason.
excluded_candidates = (
    df
    .filter(
        F.lower(F.col("brand")).isin(generic_brand_values)
        | F.col("id").isin(previously_reviewed_unidentifiable_ids)
        | F.col("title").rlike(wholesale_pattern)
    )
    .withColumn(
        "exclusion_reason",
        F.when(
            F.col("id").isin(previously_reviewed_unidentifiable_ids),
            F.lit("unidentifiable_brand")
        )
        .when(
            F.lower(F.col("brand")).isin(generic_brand_values),
            F.lit("unidentifiable_brand")
        )
        .when(
            F.col("title").rlike(wholesale_pattern),
            F.lit("wholesale")
        )
    )
    .select(
        "id",
        "brand",
        "title",
        "exclusion_reason"
    )
    .orderBy("id")
)

print(f"Listings to exclude: {excluded_candidates.count():,}")

display(excluded_candidates)

In [0]:
# Create cleaned and excluded tables

# Create the excluded listings audit table.
marketplace_excluded = (
    df
    .join(
        excluded_candidates.select("id", "exclusion_reason"),
        on="id",
        how="inner"
    )
)

# Create the cleaned dataset by removing all excluded listings.
marketplace_clean = (
    df
    .join(
        excluded_candidates.select("id"),
        on="id",
        how="left_anti"
    )
)

# Validate the results.
print(f"Original listings: {df.count():,}")
print(f"Excluded listings: {marketplace_excluded.count():,}")
print(f"Remaining listings: {marketplace_clean.count():,}")

# Save both tables.
marketplace_excluded.write.mode("overwrite").saveAsTable(
    "main.fragrance_analytics.marketplace_excluded"
)

marketplace_clean.write.mode("overwrite").saveAsTable(
    "main.fragrance_analytics.marketplace_clean"
)

In [0]:
# Validate cleaned and excluded tables

# Verify that all excluded listings were removed from the cleaned dataset.
excluded_remaining = (
    marketplace_clean
    .join(
        marketplace_excluded.select("id"),
        on="id",
        how="inner"
    )
    .count()
)

print(f"Excluded listings still present in marketplace_clean: {excluded_remaining:,}")

# Verify that the row counts reconcile.
base_count = df.count()
excluded_count = marketplace_excluded.count()
clean_count = marketplace_clean.count()

print(f"Base listings:     {base_count:,}")
print(f"Excluded listings: {excluded_count:,}")
print(f"Clean listings:    {clean_count:,}")
print(f"Clean + excluded:  {clean_count + excluded_count:,}")
print(f"Counts reconcile:   {clean_count + excluded_count == base_count}")